In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy import config
import cartopy.feature as cfeature
from cartopy.vector_transform import vector_scalar_to_grid
from matplotlib.axes import Axes
import metpy
import pint
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.units import units
import scipy as sp
from scipy.interpolate import RectBivariateSpline
import xarray as xr
from lagranto import Tra
import math
import datetime
from scipy.ndimage import rotate
import time
import iris
from iris.analysis.cartography import rotate_pole, rotate_winds
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.animation as animation
from matplotlib.animation import FFMpegWriter
from matplotlib.animation import FuncAnimation
import matplotlib.colors as mcolors

In [2]:
%run GEOS5functions.py
%matplotlib inline 

In [3]:
#variables listed here: https://opendap.nccs.nasa.gov/dods/OSSE/G5NR/Ganymed/7km/0.0625_deg/inst

# List of variable names to retrieve from the dataset
varvec = ['U', 'V', 'DELP', 'T', 'PL', 'H', 'SO4', 'CLOUD']

# Iterate over each variable name in varvec
for vind in range(len(varvec)):
    # Construct the URL for accessing the dataset corresponding to the current variable
    url = 'https://opendap.nccs.nasa.gov/dods/OSSE/G5NR/Ganymed/7km/0.0625_deg/inst/inst30mn_3d_' + varvec[vind] + '_Nv'
    
    # Open the dataset for the current variable based on the variable name
    if varvec[vind] == 'U':
        dsu = xr.open_dataset(url)  # Open dataset for 'U' (zonal wind component)
    if varvec[vind] == 'V':
        dsv = xr.open_dataset(url)  # Open dataset for 'V' (meridional wind component)
    if varvec[vind] == 'DELP':
        dsdelp = xr.open_dataset(url)  # Open dataset for 'DELP' (surface pressure and pressure thickness)
    if varvec[vind] == 'T':
        dst = xr.open_dataset(url)
    if varvec[vind] == 'PL':
        dspl = xr.open_dataset(url)
    if varvec[vind] == 'H':
        dsh = xr.open_dataset(url)
    if varvec[vind] == 'SO4':
        dsso4 = xr.open_dataset(url)



# Print a success message indicating that the datasets have been read successfully
print('success reading')

/gpfs/fs1/home/z/zilmana/modares/.virtualenvs/myenv/lib/python3.11/site-packages/xarray/coding/times.py:200: SerializationWarning: Ambiguous reference date string: 1-1-1 00:00:0.0. The first value is assumed to be the year hence will be padded with zeros to remove the ambiguity (the padded reference date string is: 0001-1-1 00:00:0.0). To remove this message, remove the ambiguity by padding your reference date strings with zeros.
  ref_date = _ensure_padded_year(ref_date)
/gpfs/fs1/home/z/zilmana/modares/.virtualenvs/myenv/lib/python3.11/site-packages/xarray/coding/times.py:200: SerializationWarning: Ambiguous reference date string: 1-1-1 00:00:0.0. The first value is assumed to be the year hence will be padded with zeros to remove the ambiguity (the padded reference date string is: 0001-1-1 00:00:0.0). To remove this message, remove the ambiguity by padding your reference date strings with zeros.
  ref_date = _ensure_padded_year(ref_date)
/gpfs/fs1/home/z/zilmana/modares/.virtualenvs/

success reading


In [4]:
coastal = np.concatenate([
    [1, 11, 20, 22, 24],
    np.arange(26, 36),
    [37, 42, 44],
    np.arange(45, 54),
    [56, 59, 60, 62, 64, 65, 67, 68, 71],
    np.arange(75, 91),
    [92, 98],
    np.arange(101, 108),
    np.arange(114, 125),
    np.arange(126, 133),
    [134, 137, 136, 138, 140, 141, 143, 144],
    np.arange(147, 165)
])
ocean = np.concatenate([
    [0],
    np.arange(2, 11),
    np.arange(12, 20),
    [21, 23, 25, 36, 38, 39, 40, 41, 43, 57, 61, 63, 66, 69, 70, 72, 73, 74, 91, 93, 94, 96, 97],
    np.arange(108, 114),
    [125, 133, 137, 146, 165]
])

all_of_all =  np.concatenate([coastal, ocean])

In [5]:
# Retrieve the list of all storms using the getallstorms function
stormlist = getallstorms()

In [6]:
all_tc = all_of_all

In [7]:
import os

for i in all_tc:
    currentstorm = i
    print(i)
    
    starttime = stormlist[3][currentstorm]
    endtime   = stormlist[4][currentstorm]
    lat1  = stormlist[5][currentstorm]
    lat2  = stormlist[6][currentstorm]
    lon1  = stormlist[7][currentstorm]
    lon2  = stormlist[8][currentstorm]
    

    stormname = stormlist[1][currentstorm][0:9]
    print(stormname)
    
    name = 'seri_so4_concentration_{stormname}_box{buffer}_futures.nc'.format(stormname=stormname, buffer=2*2.5)
    
    # Check if file exists, skip iteration if it does
    if os.path.exists(name):
        print(f"{name} already exists. Skipping...")
        continue
    
    stormtraj = np.load('/home/z/zilmana/modares/TC/trajectory/' + str(stormname) + '.npz')
    minpres = stormtraj['minpres']
    timee = stormtraj['time']
    minplat = stormtraj['minplat']
    minplon = stormtraj['minplon']
    
    timesel = timee
    print(timesel.shape)
    
    bufferrr = 2.5
    lev = dsu['lev']
    delp = dsdelp['delp']
    so4 = dsso4['so4']
    
    
    u_values, v_values, delp_values, so4_values = extract_values_with_buffer(
    dsv, dsu, delp, so4, minplat, minplon, timee, lev, buffer = bufferrr)
    
    dxlow = distance(u_values[1].isel(lat = 0, lev =71).coords['lat'].values,
                  u_values[1].isel(lon = 0, lev =71).coords['lon'].values,
                  u_values[1].isel(lat = 0, lev =71).coords['lat'].values,
                  u_values[1].isel(lon = -1, lev =71).coords['lon'].values) / u_values[1].sizes['lon'] 

    dxhigh = distance(u_values[1].isel(lat = -1, lev =71).coords['lat'].values,
                  u_values[1].isel(lon = 0, lev =71).coords['lon'].values,
                  u_values[1].isel(lat = -1, lev =71).coords['lat'].values,
                  u_values[1].isel(lon = -1, lev =71).coords['lon'].values) / u_values[1].sizes['lon'] 

    dyleft = distance(u_values[1].isel(lat = 0, lev =71).coords['lat'].values,
                  u_values[1].isel(lon = 0, lev =71).coords['lon'].values,
                  u_values[1].isel(lat = -1, lev =71).coords['lat'].values,
                  u_values[1].isel(lon = 0, lev =71).coords['lon'].values) / u_values[1].sizes['lon']

    dyright = dyleft
    
    import time
    start_time = time.time()

    total_flux = calculate_total_flux_optimized(delp_values, u_values, v_values, so4_values, 
                                            dyright, dyleft, dxhigh, dxlow)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Total runtime: {elapsed_time:.2f} seconds")
    
    total_flux.to_netcdf(name)
    print(name)


1
atl05tc02
seri_so4_concentration_atl05tc02_box5.0_futures.nc already exists. Skipping...
11
atl05tc12
seri_so4_concentration_atl05tc12_box5.0_futures.nc already exists. Skipping...
20
atl06tc04
seri_so4_concentration_atl06tc04_box5.0_futures.nc already exists. Skipping...
22
atl06tc06
seri_so4_concentration_atl06tc06_box5.0_futures.nc already exists. Skipping...
24
atl06tc08
seri_so4_concentration_atl06tc08_box5.0_futures.nc already exists. Skipping...
26
atl06tc10
seri_so4_concentration_atl06tc10_box5.0_futures.nc already exists. Skipping...
27
epc05tc01
seri_so4_concentration_epc05tc01_box5.0_futures.nc already exists. Skipping...
28
epc05tc02
seri_so4_concentration_epc05tc02_box5.0_futures.nc already exists. Skipping...
29
epc05tc03
seri_so4_concentration_epc05tc03_box5.0_futures.nc already exists. Skipping...
30
epc05tc04
seri_so4_concentration_epc05tc04_box5.0_futures.nc already exists. Skipping...
31
epc05tc05
seri_so4_concentration_epc05tc05_box5.0_futures.nc already exists. S